# A Barnes-Hut toy against direct summation

A PROTOTYPE of the compute gate (Prompt 268, COMPUTE_GATE_II.md): it is not in the catalogue and not shipped. It reads its input as the text `gravitas_input` and its settings as the JSON text `gravitas_params`, and returns `gravitas_output`, a dict of numbers. Python only, numpy and scipy only.


In [ ]:
import base64, json, math
import numpy as np
doc = json.loads(gravitas_input)
params = json.loads(gravitas_params) if gravitas_params else {}
assert doc.get("format") == "gravitas.grid" and doc.get("formatVersion") == 1, "this notebook reads gravitas.grid/1"
N = len(doc["axes"][0]["values"])
assert N <= 4000, "refused: more than 4000 bodies (the toy's cap)"
eps = float(params.get("eps", 0.05))
assert eps > 0, "refused: the softening must be positive"
raw = np.frombuffer(base64.b64decode(doc["storage"]["data"]), dtype="<f8")
px, py, pm = raw[:N].copy(), raw[N:2 * N].copy(), raw[2 * N:3 * N].copy()
G = 1.0

In [ ]:
def direct():
    dx = px[None, :] - px[:, None]
    dy = py[None, :] - py[:, None]
    w = G * pm[None, :] / (dx * dx + dy * dy + eps * eps) ** 1.5
    return (w * dx).sum(axis=1), (w * dy).sum(axis=1)

class Cell:
    __slots__ = ("m", "cx", "cy", "side", "kids", "body")

def build(idx, x0, y0, side, depth=0):
    c = Cell()
    c.side = side
    c.m = float(pm[idx].sum())
    c.cx = float((pm[idx] * px[idx]).sum() / c.m)
    c.cy = float((pm[idx] * py[idx]).sum() / c.m)
    c.kids, c.body = [], None
    if len(idx) == 1 or depth > 40:
        c.body = idx
        return c
    h = side / 2
    right = px[idx] >= x0 + h
    up = py[idx] >= y0 + h
    for r in (False, True):
        for u in (False, True):
            sub = idx[(right == r) & (up == u)]
            if len(sub):
                c.kids.append(build(sub, x0 + (h if r else 0.0), y0 + (h if u else 0.0), h, depth + 1))
    return c

root = build(np.arange(N), float(px.min()), float(py.min()), float(max(px.max() - px.min(), py.max() - py.min())) * (1 + 1e-9) + 1e-12)

def tree_accel(theta):
    ax, ay = np.zeros(N), np.zeros(N)
    count = 0
    for i in range(N):
        stack = [root]
        sx = sy = 0.0
        while stack:
            c = stack.pop()
            if c.body is not None:
                for j in c.body:
                    if j != i:
                        dx, dy = px[j] - px[i], py[j] - py[i]
                        w = G * pm[j] / (dx * dx + dy * dy + eps * eps) ** 1.5
                        sx += w * dx; sy += w * dy; count += 1
                continue
            dx, dy = c.cx - px[i], c.cy - py[i]
            d = math.sqrt(dx * dx + dy * dy)
            if theta > 0 and d > 0 and c.side / d < theta:
                w = G * c.m / (d * d + eps * eps) ** 1.5
                sx += w * dx; sy += w * dy; count += 1
            else:
                stack.extend(c.kids)
        ax[i], ay[i] = sx, sy
    return ax, ay, count

dax, day = direct()
mag = np.sqrt(dax ** 2 + day ** 2)
rows = {}
for theta in [0.0, 0.3, 0.5, 0.7, 1.0]:
    ax, ay, count = tree_accel(theta)
    rel = np.sqrt((ax - dax) ** 2 + (ay - day) ** 2) / mag
    rows[repr(theta)] = {"rms": float(np.sqrt(np.mean(rel ** 2))), "max": float(rel.max()), "interactions": count,
                         "ax": ax.tolist() if theta == 0.0 else None, "ay": ay.tolist() if theta == 0.0 else None}
gravitas_output = {"answer": {"N": N, "eps": eps, "directAx": dax.tolist(), "directAy": day.tolist(), "theta": rows}}